In [0]:
df = spark.read.format("csv")\
    .option("header","true")\
    .option("inferschema","true")\
    .load("/FileStore/tables/nyctaxi.csv") 

In [0]:
# explore columns and datatypes
# display(df)
df.columns
df.schema.fields

Out[34]: [StructField('id', StringType(), True),
 StructField('vendor_id', IntegerType(), True),
 StructField('pickup_datetime', TimestampType(), True),
 StructField('passenger_count', IntegerType(), True),
 StructField('pickup_longitude', DoubleType(), True),
 StructField('pickup_latitude', DoubleType(), True),
 StructField('dropoff_longitude', DoubleType(), True),
 StructField('dropoff_latitude', DoubleType(), True),
 StructField('store_and_fwd_flag', StringType(), True)]

In [0]:
# total no of records
df.count()

Out[8]: 625134

In [0]:
# lets save it as sql table for sql query
df.createOrReplaceTempView("taxi")
results = spark.sql("select id, count(vendor_id) from taxi group by id")
results.show()
results.count()

+---------+----------------+
|       id|count(vendor_id)|
+---------+----------------+
|id2648035|               1|
|id2111062|               1|
|id3381322|               1|
|id1309623|               1|
|id1397275|               1|
|id1415311|               1|
|id0145864|               1|
|id1720055|               1|
|id2320368|               1|
|id0365304|               1|
|id2584106|               1|
|id3732150|               1|
|id2115651|               1|
|id2395963|               1|
|id2926608|               1|
|id3903132|               1|
|id2241190|               1|
|id0753463|               1|
|id2878837|               1|
|id1969633|               1|
+---------+----------------+
only showing top 20 rows

Out[7]: 625134

In [0]:
# exploring data using aggregations
from pyspark.sql.functions import *
# separating date and time in different columns
df_date = df.withColumn('only_date',to_date("pickup_datetime"))
df_date = df_date.withColumn('only_time',date_format("pickup_datetime",'HH:mm:ss'))
# checking total no of passengers by date(which date has highest no of customers or ride)
total_passengers = df_date.groupBy("only_date").sum("passenger_count")
ridesbydate = df_date.groupBy("only_date").agg(count("id").alias("id")).orderBy("id",ascending=False)
# total_passengers.show()
# total unique no of dates
df_date.select("only_date").distinct().count()
# top 10 days with most rides
ridesbydate.show(10)

+----------+----+
| only_date|  id|
+----------+----+
|2016-04-09|4088|
|2016-03-05|4061|
|2016-04-16|4049|
|2016-05-21|4006|
|2016-05-07|4002|
|2016-03-12|4002|
|2016-05-06|3995|
|2016-04-02|3988|
|2016-05-14|3966|
|2016-04-15|3961|
+----------+----+
only showing top 10 rows



In [0]:
ridesbytime = df_date.groupBy(date_format("pickup_datetime",'HH').alias("only_time")).agg(count("id").alias("id")).orderBy("id",ascending=False)
ridesbytime.show()

+---------+-----+
|only_time|   id|
+---------+-----+
|       18|38841|
|       19|38437|
|       20|36295|
|       21|35746|
|       22|34567|
|       17|32916|
|       14|32345|
|       15|31000|
|       13|30524|
|       12|30448|
|       23|29871|
|       11|29145|
|       09|29127|
|       08|28854|
|       10|27997|
|       16|27714|
|       07|23818|
|       00|22714|
|       01|16529|
|       06|14122|
+---------+-----+
only showing top 20 rows



In [0]:
# try to find peak hours group 

from pyspark.sql.functions import when

df_date = df_date.withColumn('only_time',date_format("pickup_datetime",'HH:mm:ss'))
df_hours = df_date.withColumn("hour_group", \
    when((df_date.only_time >='00:00:00') & (df_date.only_time<'06:00:00'), '00-06')\
   .when((df_date.only_time >='06:00:00') & (df_date.only_time<'12:00:00'), '06-12')\
   .when((df_date.only_time >='12:00:00') & (df_date.only_time<'18:00:00'), '12-18')\
   .when((df_date.only_time >='18:00:00') & (df_date.only_time<'24:00:00'), '18-24')\
    )

ridesbygrouphours = df_hours.groupBy("hour_group").agg(count("id").alias("total_rides")).orderBy("total_rides",ascending=False)
ridesbygrouphours.show()

+----------+-----------+
|hour_group|total_rides|
+----------+-----------+
|     18-24|     213757|
|     12-18|     184947|
|     06-12|     153063|
|     00-06|      73367|
+----------+-----------+



In [0]:
# import matplotlib.pyplot as plt
# plt.bar(ridesbygrouphours.hour_group,ridesbygrouphours.total_rides)
# plt.show()
# ridesbygrouphours.schema.fields
from pyspark.sql.types import StringType, BooleanType, IntegerType 

ridesbygrouphours = ridesbygrouphours.select(ridesbygrouphours.hour_group,
    (ridesbygrouphours.total_rides.cast(IntegerType())))
ridesbygrouphours.show()    
# import matplotlib.pyplot as plt
# plt.bar(ridesbygrouphours.hour_group,ridesbygrouphours.total_rides)
# plt.show() 

+----------+-----------+
|hour_group|total_rides|
+----------+-----------+
|     18-24|     213757|
|     12-18|     184947|
|     06-12|     153063|
|     00-06|      73367|
+----------+-----------+

